# AmsterdamUMCdb Exploration

## Capstone Goal
Predict successful weaning from mechanical ventilation using time-series deep learning models.

## Objectives
- Understand database structure
- Identify ventilation-related tables
- Find extubation events
- Explore physiological measurements
- Understand timestamps and patient identifiers

### Imports

In [ ]:
import pandas as pd
import numpy as np
from google.cloud import bigquery
import matplotlib.pyplot as plt

### Retrieving Google Project Id

In [ ]:
client = bigquery.Client()

print("Connected to BigQuery")

In [ ]:
# sets *your* project id
PROJECT_ID = "capstoneweaningprediction" #@param {type:"string"}

# Sets the default BigQuery dataset for accessing AmsterdamUMCdb

If you have received instructions to use a specific BigQuery instance, change the default settings here. Otherwise use these default values.

In [ ]:
# sets default dataset for AmsterdamUMCdb
DATASET_PROJECT_ID = 'amsterdamumcdb' #@param {type:"string"}
DATASET_ID = 'version1_5_0' #@param {type:"string"}
LOCATION = 'eu' #@param {type:"string"}

# Provide your credentials to access the AmsterdamUMCdb dataset on Google BigQuery
Authenticate your credentials with Google Cloud Platform and set your default Google Cloud Project ID as an environment variable for running query jobs.

1. Run the cell. The `Allow this notebook to access your Google credentials?` prompt appears. Select `Allow`.
2. In the `Sign in - Google Accounts` dialog, use the account you registered during the AmsterdamUMCdb application process and select `Allow` again.

In [ ]:
import os
from google.colab import auth

# all libraries check this environment variable, so set it:
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID

auth.authenticate_user()
print('Authenticated')

# Enable data table display

Colab includes the `google.colab.data_table` package that can be used to display Pandas dataframes as an interactive data table (default limits: `max_rows = 20000`, `max_columns = 20`). This is especially useful when exploring the  tables or dictionary from AmsterdamUMCdb. It can be enabled with:

In [ ]:
%load_ext google.colab.data_table
from google.colab.data_table import DataTable

# change default limits:
DataTable.max_columns = 50
DataTable.max_rows = 80000


## Set the default query job configuration for magics

In [ ]:
%load_ext bigquery_magics
from bigquery_magics import bigquery_magics
from google.cloud import bigquery

# sets the default query job configuration
def_config = bigquery.job.QueryJobConfig(default_dataset=DATASET_PROJECT_ID + "." + DATASET_ID)
bigquery_magics.context.default_query_job_config = def_config

## Query the `person` table and copy the data to the `persons` Pandas dataframe:

The `person` table contains a record for each patient in AmsterdamUMCdb.

Since this is a relatively small table, it is acceptable to use `SELECT *`.

**Note**: Should an error occur while running the query, please see
the AmsterdamUMCdb BigQuery [Frequently Asked Questions](https://github.com/AmsterdamUMC/AmsterdamUMCdb/wiki/bigquery#faq).

In [ ]:
%%bigquery person
SELECT * FROM `amsterdamumcdb.version1_5_0.person`;


## Set the default query job configuration for google-cloud-bigquery client

In [ ]:
from google.cloud import bigquery

# BigQuery requires a separate config to prevent the 'BadRequest: 400 Cannot explicitly modify anonymous table' error message
job_config = bigquery.job.QueryJobConfig()

# sets default client settings by re-using the previously defined config
client = bigquery.Client(project=PROJECT_ID, location=LOCATION, default_query_job_config=def_config)

### Explore Concept Table

In [ ]:
query = f"""
SELECT *
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
LIMIT 10
"""

concept_df = client.query(query).to_dataframe()

concept_df.head()

In [ ]:
concept_df.shape

#### Explore Domain Distribution

In [ ]:
query = f"""
SELECT domain_id, COUNT(*) AS count
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
GROUP BY domain_id
ORDER BY count DESC
"""

concept_domain_df = client.query(query).to_dataframe()

concept_domain_df

Measurement, Device, and Procedure — the three domains most relevant to the project — are well represented with 186K, 236K, and 99K concepts respectively. Drug dominates the vocabulary at 4.6M, but that's less central here. The data has solid coverage where it counts for this analysis.

#### Explore Vocabulary Distribution

In [ ]:
query = f"""
SELECT vocabulary_id, COUNT(*) AS count
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
GROUP BY vocabulary_id
ORDER BY count DESC
"""

concept_vocabulary_df = client.query(query).to_dataframe()

concept_vocabulary_df

RxNorm Extension and RxNorm together dominate the vocabulary — largely driving that 4.RxNorm Extension and RxNorm together dominate the vocabulary — largely driving that 4.6M Drug count seen earlier. For the project's focus areas, SNOMED (1M) and LOINC (269K) are the key standardized vocabularies covering Measurement, Device, and Procedure concepts, while OMOP Extension is relatively small at 1.4K — meaning most relevant concepts here follow well-established international standards.

#### Search Ventilation Concepts

In [ ]:
query = f"""
SELECT concept_id, concept_name, domain_id
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
WHERE concept_name LIKE '%ventilation%'
"""

ventilation_concept_df = client.query(query).to_dataframe()

ventilation_concept_df


Ventilation concepts here are remarkably broad — covering everything from mechanical ventilation modes like SIMV and CPAP to weaning protocols, hypoventilation disorders, and even ear tube insertions. The depth in invasive vs non-invasive ventilation procedures makes this particularly useful for the project, though the mix of clinical and non-clinical entries like HVAC job titles shows the vocabulary casts a wide net.

#### Search Extubation Concepts

In [ ]:
query = f"""
SELECT concept_id, concept_name, domain_id
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
WHERE concept_name LIKE '%extubation%'
"""

extubation_concept_df = client.query(query).to_dataframe()

extubation_concept_df

Extubation concepts here are fairly focused — mostly around timing, accidental removal, and difficulty during the process. The standout clinically is post-extubation respiratory failure requiring reintubation, which is a key risk indicator worth tracking closely in any airway management analysis.

#### Search Intubation Concepts

In [ ]:
query = f"""
SELECT concept_id, concept_name, domain_id
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
WHERE concept_name LIKE '%intubation%'
"""

intubation_concept_df = client.query(query).to_dataframe()

intubation_concept_df

The intubation concepts span a solid range — procedures like tracheal, nasotracheal, and fiberoptic intubation sit alongside devices like video laryngoscopes and esophageal detectors, with observations capturing difficult or failed intubation scenarios. There's also good clinical depth here, covering everything from emergency intubation to post-intubation complications like subglottic stenosis and acute respiratory failure.

### Identify Respiratory Measurements

In [ ]:
query = f"""
SELECT concept_id, concept_name, domain_id
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
WHERE lower(concept_name) LIKE '%respiratory%'
"""

respiratory_concept_df = client.query(query).to_dataframe()

respiratory_concept_df

The result contains a list of respiratory inhalation gas concepts, all variations of medical oxygen. Here's the interpretation:
This is a vocabulary mapping file containing concept IDs for medical oxygen used as a respiratory inhalation gas, with two main strength variants — oxygen 995 mL/L and oxygen 99 L/100 L — both representing high-concentration oxygen administered by inhalation for respiratory therapy. The thousands of rows are different concept-ID entries (likely from a clinical terminology like OMOP/RxNorm) all pointing to the same therapeutic substance, indicating a standardised drug-concept reference list for inhaled oxygen.

#### Explore Peep Concepts

In [ ]:
query = f"""
SELECT concept_id, concept_name, domain_id
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
WHERE lower(concept_name) LIKE '%peep%'
"""

peep_concept_df = client.query(query).to_dataframe()

peep_concept_df

Most of these are clinical PEEP (Positive End-Expiratory Pressure) concepts used in ventilator management — covering the device hardware (reusable and single-use PEEP valves), the different measured forms (intrinsic, extrinsic, total, dynamic), and clinical actions like adjusting PEEP to its optimal or best level. A handful of unrelated entries also slipped in — "Peep Hill," "Peeping Tom," "Spring peeper," and Peeps-branded hand sanitiser — which are name collisions rather than respiratory concepts.

#### Explore Mechanical Ventilation Concepts

In [ ]:
query = f"""
SELECT concept_id, concept_name, domain_id
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
WHERE lower(concept_name) LIKE '%mechanical ventilation%'
"""

mechanical_ventilation_concept_df = client.query(query).to_dataframe()

mechanical_ventilation_concept_df

These are mechanical ventilation concepts covering both invasive and non-invasive forms, with subtypes broken down by duration (under or over 96 hours, continuous, unspecified) and by clinical context such as admission status, weaning response, and liberation at discharge. A few entries are adjacent rather than core — disposable moisture-exchanger accessories, home visits for ventilator care, and tocilizumab drug codes that simply mention ventilated COVID patients in their description.

### Explore the Device Cocepts

In [ ]:
query = f"""
SELECT concept_id, concept_name, domain_id
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
WHERE lower(concept_name) LIKE '%device%'
"""

device_concept_df = client.query(query).to_dataframe()

device_concept_df

This file is a catalogue of medical device concepts spanning a wide clinical range — assistive listening aids, prosthetics and terminal hand devices, intrauterine and vaginal drug-delivery devices, cochlear implant accessories, wound-therapy and sterilisation equipment, and rehabilitation training devices. A large portion also covers device-related complications such as mechanical failure, obstruction, infection, loosening, and inflammation linked to implanted or retained devices, suggesting the list combines both device products and the clinical events associated with their u

### Explore Procedure Concepts

In [ ]:
query = f"""
SELECT concept_id, concept_name, domain_id
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
WHERE domain_id LIKE '%Procedure%'
"""

procedure_concept_df = client.query(query).to_dataframe()

procedure_concept_df

### Build Candidate concept dictionary

#### Pick Mechanical Ventilation root ID

In [ ]:
mv_row = mechanical_ventilation_concept_df[
    (mechanical_ventilation_concept_df['concept_name'].str.lower() == 'mechanical ventilation')
    & (mechanical_ventilation_concept_df['domain_id'] == 'Procedure')
]
mv_id = int(mv_row['concept_id'].iloc[0])
print('Mechanical ventilation:', mv_id)
mv_row

#### Pick Peep root id

In [ ]:
peep_row = peep_concept_df[
    (peep_concept_df['concept_name'] == 'PEEP Respiratory system')
    & (peep_concept_df['domain_id'] == 'Measurement')
]
peep_id = int(peep_row['concept_id'].iloc[0])
print('PEEP:', peep_id)
peep_row

#### Pick extubation root ID

In [ ]:
ext_row = extubation_concept_df[
    (extubation_concept_df['concept_name'] == 'Recently performed an extubation of trachea') &
    (extubation_concept_df['domain_id'] == 'Observation')
]

ext_id = int(ext_row['concept_id'].iloc[0])

print('Extubation:', ext_id)
ext_row

#### Pick intubation root ID

In [ ]:
int_row = intubation_concept_df[
    (intubation_concept_df['concept_name'] == 'Unintended endobronchial intubation') &
    (intubation_concept_df['domain_id'] == 'Procedure')
]

int_id = int(int_row['concept_id'].iloc[0])

print('Intubation:', int_id)

int_row